In [0]:
%fs ls /Volumes/gizmobox/landing/operational_data/memberships/

In [0]:
select * from binaryFile.`/Volumes/gizmobox/landing/operational_data/memberships/*/*.png`;

In [0]:
create or replace view gizmobox.bronze.v_memberships as select * from binaryFile.`/Volumes/gizmobox/landing/operational_data/memberships/*/*.png`;

In [0]:
select * from gizmobox.bronze.v_memberships;

In [0]:
%python
from pyspark.sql.functions import *

df = spark.read.format("binaryFile").load("/Volumes/gizmobox/landing/operational_data/memberships/*/*.png")

from PIL import Image
import io
import matplotlib.pyplot as plt

rows = df.select("content").collect()
for r in rows:
    img = Image.open(io.BytesIO(r.content))
    plt.imshow(img)
    plt.axis('off')
    plt.show()



In [0]:
SHOW VOLUMES IN gizmobox.landing;

In [0]:
%python
 
# Read the memberships data to a dataframe
memberships_df = (spark.read.format("binaryFile").load("/Volumes/gizmobox/landing/operational_data/memberships/*/*.png"))
 
# Write the memberships data to a managed bronze table
memberships_df.write.saveAsTable("gizmobox.bronze.memberships")

In [0]:
SELECT * FROM gizmobox.bronze.memberships;

In [0]:
%python
from pyspark.sql.functions import udf
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from PIL import Image
import io

def get_image_info(content):
    try:
        img = Image.open(io.BytesIO(content))
        return (img.width, img.height, img.format)
    except:
        return (None, None, None)

schema = StructType([
    StructField("width", IntegerType()),
    StructField("height", IntegerType()),
    StructField("format", StringType())
])

image_info_udf = udf(get_image_info, schema)

image_df = df.withColumn("image_info", image_info_udf("content")) \
    .select(
        "path",
        "content",
        "image_info.width",
        "image_info.height",
        "image_info.format"
    )